In [ ]:
import pandas as pd
import pypsa

### Paths

In [ ]:
output_fn = "../../results/supplementary/cost_value_differences_with_usage.csv"

### Network comparison

In [ ]:
test = pd.read_csv("../../results/supplementary/cost_value_differences_with_usage.csv")

In [ ]:
n_old = pypsa.Network(
    "../../results/old_costs/cost_year~2050/transport_cost~irena/demand~0.2/network.nc"
)

In [ ]:
n_new = pypsa.Network(
    "../../results/cost_year~2050/transport_cost~irena/demand~0.2/network.nc"
)

In [ ]:
# Compare the two networks

n_old == n_new

In [ ]:
# Compare the two networks

n_old.equals(n_new, log_mode="verbose")

### Comparison of trade network marginal costs of generators

In [ ]:
n_old.generators.marginal_cost.describe()

In [ ]:
n_new.generators.marginal_cost.describe()

### Compare two cost files

In [ ]:
cost_old = pd.read_csv("../../data/technology_data/costs_2050.csv", index_col=0)
cost_new = pd.read_csv("../../resources/technology_data/costs_2050.csv", index_col=0)

In [ ]:
# Load the cost data
cost_df = pd.read_csv("../../data/technology_data/costs_2050.csv")

# List of technologies used in your script (from the code you provided)
used_techs = {
    "onwind",
    "solar-utility",
    "battery storage",
    "hydrogen storage tank incl. compressor",
    "electrolysis",
    # "offwind"  # commented out in your script, so not active
}

# Add a boolean column if technology is used in script
cost_df["used_in_script"] = cost_df["technology"].isin(used_techs)

# Now prepare for comparison: set multiindex on technology and parameter
cost_df.set_index(["technology", "parameter"], inplace=True)

# Load new costs file (for comparison)
cost_new = pd.read_csv("../../resources/technology_data/costs_2050.csv")
cost_new["used_in_script"] = cost_new["technology"].isin(used_techs)
cost_new.set_index(["technology", "parameter"], inplace=True)

# Extract and align 'value' columns
old_values = cost_df["value"]
new_values = cost_new["value"]
old_values, new_values = old_values.align(new_values, join="outer")

# Compute difference
diff = new_values - old_values

# Assemble comparison DataFrame
differences = pd.DataFrame({"old": old_values, "new": new_values, "delta": diff})

# Bring the 'used_in_script' flags (technology level) from old and new data
# Since 'used_in_script' depends only on 'technology', take first level index and map
differences = differences.reset_index()
differences["used_in_script_old"] = differences["technology"].map(
    cost_df.reset_index()
    .drop_duplicates("technology")
    .set_index("technology")["used_in_script"]
)
differences["used_in_script_new"] = differences["technology"].map(
    cost_new.reset_index()
    .drop_duplicates("technology")
    .set_index("technology")["used_in_script"]
)

# Add relative change (%) column
differences["relative_change (%)"] = 100 * differences["delta"] / differences["old"]

# Optional: filter by tolerance
tolerance = 1e-3
differences = differences[differences["delta"].abs() > tolerance]

# Save result with usage flags
differences.to_csv(output_fn, index=False)